# 🚀 SpaceX Falcon 9 Landing Prediction
## Notebook 6 — Launch Site Geography with Folium

Location matters. We visualise all SpaceX launch sites on an interactive map,
overlay success/failure markers, and measure distances to nearby infrastructure.

**Tasks:**
1. Mark all launch sites on a map
2. Colour-code success/failure launches per site
3. Calculate great-circle distances from a site to nearby landmarks


In [6]:
%pip install folium wget pandas -q
import folium
import wget
import pandas as pd
import math
from folium.plugins import MarkerCluster
from folium.features import DivIcon


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
# IBM-hosted geo-enriched dataset (lat/long already attached)
geo_df = pd.read_csv(
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud"
    "/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv"
)
geo_df = geo_df[['Launch Site', 'Lat', 'Long', 'class']]
print(f"Shape: {geo_df.shape}")
geo_df.head()


Shape: (56, 4)


,Launch Site,Lat,Long,class
0,CCAFS LC-40,28.562302,-80.577356,0
1,CCAFS LC-40,28.562302,-80.577356,0
2,CCAFS LC-40,28.562302,-80.577356,0
3,CCAFS LC-40,28.562302,-80.577356,0
4,CCAFS LC-40,28.562302,-80.577356,0


### Task 1 — Mark each launch site with a labelled circle

In [8]:
launch_sites = (
    geo_df.groupby('Launch Site')[['Lat', 'Long']]
          .mean()
          .reset_index()
)

site_map = folium.Map(location=[28.5, -80.5], zoom_start=5)

for _, row in launch_sites.iterrows():
    folium.Circle(
        location=[row['Lat'], row['Long']],
        radius=1000,
        color='navy', fill=True, fill_opacity=0.35,
        tooltip=row['Launch Site']
    ).add_to(site_map)
    folium.map.Marker(
        location=[row['Lat'], row['Long']],
        icon=DivIcon(
            icon_size=(160, 36),
            icon_anchor=(0, 0),
            html=(
                f'<div style="font-size:11px; color:darkblue; '
                f'font-weight:bold;">{row["Launch Site"]}</div>'
            )
        )
    ).add_to(site_map)

site_map


### Task 2 — Success/failure cluster markers per site

In [9]:
outcome_map = folium.Map(location=[28.5, -80.5], zoom_start=5)
cluster = MarkerCluster().add_to(outcome_map)

for _, row in geo_df.iterrows():
    color = 'green' if row['class'] == 1 else 'red'
    label = 'Success' if row['class'] == 1 else 'Failure'
    folium.CircleMarker(
        location=[row['Lat'], row['Long']],
        radius=5,
        color=color, fill=True, fill_color=color, fill_opacity=0.75,
        popup=folium.Popup(
            f"<b>{row['Launch Site']}</b><br>Outcome: {label}",
            max_width=200
        )
    ).add_to(cluster)

outcome_map


### Task 3 — Great-circle distances from CCAFS SLC-40

We compute how far CCAFS SLC-40 is from:
- Cocoa Beach (nearest city)
- The Atlantic coastline
- The US-1 highway


In [10]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in kilometres using the Haversine formula."""
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = (math.sin(dphi / 2) ** 2
         + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2) ** 2)
    return 2 * R * math.asin(math.sqrt(a))

# CCAFS SLC-40
site_lat, site_lon = 28.5620, -80.5772

landmarks = {
    "Cocoa Beach (nearest city)": (28.3200, -80.6076),
    "Atlantic coastline":         (28.5620, -80.5200),
    "US-1 Highway":               (28.5620, -80.5900),
}

dist_map = folium.Map(location=[site_lat, site_lon], zoom_start=12)
folium.Marker(
    [site_lat, site_lon],
    popup="CCAFS SLC-40",
    icon=folium.Icon(color='blue', icon='rocket', prefix='fa')
).add_to(dist_map)

for name, (lat, lon) in landmarks.items():
    dist = haversine_km(site_lat, site_lon, lat, lon)
    folium.Marker(
        [lat, lon],
        popup=f"{name}<br><b>{dist:.2f} km</b> from SLC-40",
        icon=folium.Icon(color='red', icon='info-sign')
    ).add_to(dist_map)
    folium.PolyLine(
        locations=[[site_lat, site_lon], [lat, lon]],
        color='gray', weight=2, dash_array='6 4',
        tooltip=f"{dist:.2f} km"
    ).add_to(dist_map)
    print(f"  {name}: {dist:.2f} km")

dist_map


  Cocoa Beach (nearest city): 27.07 km
  Atlantic coastline: 5.59 km
  US-1 Highway: 1.25 km
